# Experiment 1 — headline benchmark (PD)

5-fold CV on the PD datasets (tuned/HPO models). For every matrix metric — **AUC**, **Brier**, **F1** — the **same three views**: dataset×method **matrix** (best column left), per-method **bar** (value above; fold-level std error bars), and a **box** plot of the across-dataset spread. The **rank by AUC** (kept right after AUC) gets the same three views. Then the **effect of HPO**, a **time analysis** (train + predict + HPO), and a summary table. Foundation-model names are **red**. Figures → `figures/experiment1/pd/`.

In [ ]:
import sys
from pathlib import Path
PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from src.utils.paths import results_root
from src.methods.method_config import HPO_METHODS          # tuned methods -> HPO time x n_trials
from src.visualizations.experiment_plots import (
    apply_style, reset_figure_dir, load_summary,
    performance_heatmap, metric_bars, metric_boxplots,
    rank_heatmap, method_ranking_bars, rank_boxplots,
    hpo_improvement_bars, compute_time_bars, compute_time_boxplot,
    runtime_performance_scatter, foundation_vs_baseline_size_trend,
    foundation_vs_baseline_scatter, pd_summary_text, lgd_summary_text,
)
apply_style()

RESULTS_ROOT = results_root()          # override here if your copy lives elsewhere
SUMMARY_DIR  = RESULTS_ROOT / 'summaries'
FIGURES_DIR  = reset_figure_dir(PROJECT_ROOT / 'figures' / 'experiment1/pd')
print('results:', RESULTS_ROOT, '| figures:', FIGURES_DIR)

In [ ]:
df      = load_summary(SUMMARY_DIR, experiment='experiment1', task='pd', aggregated=False, hpo_mode='HPO')   # tuned models + their times
df_both = load_summary(SUMMARY_DIR, experiment='experiment1', task='pd', aggregated=False, hpo_mode=None)    # HPO vs NO_HPO, for the HPO-effect bars
print(f'{df["method"].nunique()} methods x {df["dataset"].nunique()} datasets')

## 1. AUC — discrimination

Matrix · per-method bar (fold-level std) · across-dataset box.

In [ ]:
performance_heatmap(df, 'AUC', task_name='PD', higher_is_better=True, out_dir=FIGURES_DIR)
metric_bars(df, 'AUC', task_name='PD', higher_is_better=True, out_dir=FIGURES_DIR)
metric_boxplots(df, 'AUC', task_name='PD', higher_is_better=True, out_dir=FIGURES_DIR)

## 2. Rank by AUC (1 = best on that dataset)

The per-dataset ranking that underlies the AUC table — kept right next to it. Matrix · mean-rank bar · rank box.

In [ ]:
rank_heatmap(df, 'AUC', task_name='PD', out_dir=FIGURES_DIR)
method_ranking_bars(df, 'AUC', task_name='PD', out_dir=FIGURES_DIR)
rank_boxplots(df, 'AUC', task_name='PD', out_dir=FIGURES_DIR)

## 3. Brier score — calibration (lower is better)

Matrix · bar · box.

In [ ]:
performance_heatmap(df, 'Brier', task_name='PD', higher_is_better=False, out_dir=FIGURES_DIR)
metric_bars(df, 'Brier', task_name='PD', higher_is_better=False, out_dir=FIGURES_DIR)
metric_boxplots(df, 'Brier', task_name='PD', higher_is_better=False, out_dir=FIGURES_DIR)

## 4. F1 — decision quality

Matrix · bar · box.

In [ ]:
performance_heatmap(df, 'F1', task_name='PD', higher_is_better=True, out_dir=FIGURES_DIR)
metric_bars(df, 'F1', task_name='PD', higher_is_better=True, out_dir=FIGURES_DIR)
metric_boxplots(df, 'F1', task_name='PD', higher_is_better=True, out_dir=FIGURES_DIR)

## 5. Effect of HPO on AUC

Mean HPO-minus-NO_HPO change per method. Foundation models are 0 by design (their HPO run copies NO_HPO).

In [ ]:
hpo_improvement_bars(df_both, 'AUC', task_name='PD', out_dir=FIGURES_DIR)

## 6. Time analysis — train + predict + HPO

For the tunable methods the training time is multiplied by the number of HPO trials (`n_trials`) so the bars reflect the **hyperparameter-search** cost as well as the final fit + predict; in-context foundation models are unchanged. The boxplot shows the raw per-fold compute-time spread, and the scatter is the accuracy-vs-cost frontier (**median** time on x, **mean** AUC on y).

In [ ]:
compute_time_bars(df, task_name='PD', hpo_methods=HPO_METHODS, n_trials=20, out_dir=FIGURES_DIR)
compute_time_boxplot(df, task_name='PD', out_dir=FIGURES_DIR)

In [ ]:
runtime_performance_scatter(df, 'AUC', task_name='PD', hpo_methods=HPO_METHODS, n_trials=20, out_dir=FIGURES_DIR)

## 7. TabPFN v3 vs CatBoost — head-to-head

First, the **relative AUC gain** of TabPFN v3 over CatBoost per dataset against dataset size, with an OLS trend — does the edge grow on small data? Green = TabPFN v3 wins; the dashed line is equal performance. Then a per-dataset AUC scatter of the two methods with the **y = x** equal-performance diagonal.

In [ ]:
foundation_vs_baseline_size_trend(df, metric='AUC', task='pd', task_name='PD', relative=True, higher_is_better=True, out_dir=FIGURES_DIR)

In [ ]:
foundation_vs_baseline_scatter(df, metric='AUC', task_name='PD', higher_is_better=True, out_dir=FIGURES_DIR)

## 8. Summary

In [ ]:
pd_summary_text(df, task_name='Experiment 1 — PD')